In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.statespace.sarimax import SARIMAX
import optuna


c:\Users\Marwa\anaconda3\envs\cc_abs\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [53]:
# Show all rows
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_rows', 1000)

- DRCC =  Credit Card Delinquency rate (all commercial banks)
- COR = Credit card charge-off rate
- TERM = Terms on Credit Card plans

In [54]:
corc_df = pd.read_csv('../data/CORCCACBS.csv')
drcc_df = pd.read_csv('../data/DRCCLACBS.csv')
term_df = pd.read_csv('../data/TERMCBCCALLNS.csv')

In [ ]:
print(corc_df.head(15))
print(drcc_df[drcc_df['date']>'1994'])
print(term_df.head(15))

Organize our data - it currently has multiple entries for each date + has a hours/minutes/second. I'll make a function to make this more streamlined

In [55]:

def fix_df(df):

    a=df.groupby(df['date'])['value'].mean().round(2)
    
    b = pd.to_datetime(df['date'].unique())

    df = pd.DataFrame({
        'date':b,
        'value':a.values
    })

    df = df.set_index('date')

    return df



In [56]:
drcc_df = fix_df(drcc_df)
corc_df = fix_df(corc_df)
term_df = fix_df(term_df)
term_df = term_df.dropna()

In [ ]:
print(drcc_df.head(1))
print(corc_df.head(1))
print(term_df.head(1))
print(len(drcc_df),len(corc_df),len(term_df))

Plotting them out 
- They all start at different dates, so I'll start them all at 1995 to have an equal starting date for easier comparison

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

drcc_df.loc['1995':]['value'].plot(ax=axes[0], title='Delinquency Rate (DRCCLACBS)', color='steelblue')
corc_df.loc['1995':]['value'].plot(ax=axes[1], title='Charge-off Rate (CORCCACBS)', color='coral')
term_df.loc['1995':]['value'].plot(ax=axes[2], title='Credit Card Terms (TERMCBCCALLNS)', color='green')

plt.tight_layout()
plt.show()

Finding correlational value between variables vs. outcome

In [57]:
print(len(drcc_df.loc['1995':]['value']),drcc_df.loc['1995':]['value'].tail(4))
print(len(corc_df.loc['1995':]['value']),corc_df.loc['1995':]['value'].tail(4))
print(len(term_df.loc['1995':'2025']['value']),term_df.loc['1995':]['value'].tail(4))


term_corr = term_df.loc['1995':]['value'].ffill().asfreq('QS').last()
drcc_corr = drcc_df.loc['1995':]['value'].asfreq('QS').last()



print(term_corr.isna().sum())



print('\n','The Correlation between Charge off rate and Delinquency rate is:',drcc_df.loc['1995':]['value'].corr(corc_df.loc['1995':]['value']))
print('\n','Correlation between delinquency and term length is:',term_corr.corr(drcc_corr))



#Figure out a way to match

124 date
2025-01-01    3.06
2025-04-01    3.04
2025-07-01    2.98
2025-10-01    2.94
Name: value, dtype: float64
124 date
2025-01-01    4.43
2025-04-01    4.18
2025-07-01    4.18
2025-10-01    4.11
Name: value, dtype: float64
124 date
2025-05-01    21.16
2025-08-01    21.39
2025-11-01    20.97
2026-02-01    21.00
Name: value, dtype: float64


AttributeError: 'Series' object has no attribute 'last'

We have a time mismatch clearly, how should I split?
A couple of ideas:

- Align our data to yearly data, that way we'll have more aligned data points but probably not the best idea will be mising out on (4!!) predictions per year, since the data we're looking for, the delinquency rate, is calculated quarterly.

- try to match where we can? maybe align closes data to that data point ??

- let Pandas try to match the timelines as best it can

Let pandas merge

In [59]:
df_multi = pd.concat([drcc_df,corc_df,term_df], axis =1 , join='outer')

C:\Users\Marwa\AppData\Local\Temp\ipykernel_9424\428149228.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_multi = pd.concat([drcc_df,corc_df,term_df], axis =1 , join='outer')


In [60]:
# we chose 1995, since thats the earliest date that they all have data

df_multi = df_multi.loc["1995":]
print(len(df_multi),df_multi.head(10))


df_multinfiltered = df_multi.asfreq('QS').dropna()

print(len(df_multinfiltered))


#this had a much higher count since term had data starting at a much earlier date than the other two.

249             value  value  value
date                           
1995-01-01   3.48   2.96    NaN
1995-02-01    NaN    NaN  16.10
1995-04-01   3.67   3.29    NaN
1995-05-01    NaN    NaN  16.14
1995-07-01   3.86   3.66    NaN
1995-08-01    NaN    NaN  15.92
1995-10-01   3.96   3.93    NaN
1995-11-01    NaN    NaN  15.81
1996-01-01   4.04   4.23    NaN
1996-02-01    NaN    NaN  15.80
0


without term this time:
also Changing the name of Column from value to distinct name 

In [61]:
drcc_df.rename(columns={'value': 'delinquency'}, inplace=True)
corc_df.rename(columns={'value': 'charge_off'}, inplace=True)

In [62]:
df_multi1 = pd.concat([drcc_df,corc_df], axis =1 , join='outer')
df_multi1=df_multi1.asfreq('QS').dropna()
print(len(df_multi1),df_multi1.head(10))

140             delinquency  charge_off
date                               
1991-01-01         5.27        4.19
1991-04-01         5.44        4.63
1991-07-01         5.37        4.81
1991-10-01         5.31        4.63
1992-01-01         5.28        4.91
1992-04-01         5.06        4.84
1992-07-01         5.00        4.35
1992-10-01         4.69        4.50
1993-01-01         4.61        4.11
1993-04-01         4.43        3.95


C:\Users\Marwa\AppData\Local\Temp\ipykernel_9424\2420141979.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_multi1 = pd.concat([drcc_df,corc_df], axis =1 , join='outer')


## TERMCBCCALLNS — dropped

- Excluded due to time cycle mismatch -> quarterly cycle (QS-NOV vs QS-JAN) 
- data loss on merge (249 to 0 observations) after dropping Null



#### Next, we'll split into train val test

also introduce a feature engineered variable, 'anamoly flag'

In [63]:
def traintestsplit(df):
    train = df.loc[:'2015']
    val = df.loc['2016':'2019']
    test = df.loc['2020':]

    return train,val ,test

def featureng(df):
    df_fe = df.copy()
    df_fe['anamoly_flag'] = ((df_fe['charge_off'] > 5) | 
                          (df_fe['charge_off'] < 2.0)).astype(int)
    return df_fe

df_multi1_fe = featureng(df_multi1)
# this only includes delinquency rates as well as charge off rates. Now we'll split
train_mv, val_mv, test_mv = traintestsplit(df_multi1_fe)





In [64]:
print(len(train_mv[train_mv['anamoly_flag']==1])) # 35 points in time where we had 'anamoly' rates!

35


##### SARIMAX with Exogenous features

In [67]:
def objective(trial):
    
    p_= trial.suggest_int("p", 0, 3)
    d_= trial.suggest_int("d", 0, 2)
    q_= trial.suggest_int("q", 0, 3)
    P_= trial.suggest_int("P", 0, 2)
    D_= trial.suggest_int("D", 0, 1)
    Q_= trial.suggest_int("Q", 0, 2)
    

    try:
        model = SARIMAX(train_mv['charge_off'],
                        exog=train_mv[['anamoly_flag','delinquency']],
                        order=(p_,d_,q_),
                        seasonal_order=(P_,D_,Q_,4),
                        enforce_stationarity=False,
                        enforce_invertibility=False)


        result = model.fit(disp=False,maxiter=200,method='powell')

        #forecasting over the val timeline

        y_pred = result.forecast(steps=len(val_mv),
                                 exog=val_mv[['anamoly_flag', 'delinquency']])
        y_pred.index = val_mv.index

        mae = (y_pred - val_mv['charge_off']).abs().mean()
        return mae

    except Exception:
        return 999


study = optuna.create_study(direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=50, timeout=600)

print("Best MAE:", f'{study.best_value:.4f}')
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Best MAE: 0.1778
Best params:
  p: 2
  d: 1
  q: 1
  P: 2
  D: 1
  Q: 1


Best MAE: 0.1778
Best params:
  p: 2
  d: 1
  q: 1
  P: 2
  D: 1
  Q: 1